In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#pip install mne             # To be installed if not already done
#pip install h5py
#pip install joblib

import numpy as np
import os
import h5py
import multiprocessing as mp
from joblib import Parallel, delayed
import re
from collections import defaultdict

from hdf5_creation import prepare_for_hdf5, update_hdf5

In [ ]:
epoch_length = 10 #in seconds

path_to_mat = r"D:\dilon_data\mat_collection\test_similar_layout_both_datasets"
hdf5_path = r"D:\dilon_data\hdf5_collection\complete_dataset.h5"

# try:
#     Database = h5py.File(hdf5_path, 'w')  # Output directory path
# except:
#     Database = h5py.File(hdf5_path, 'a')

files = np.ravel(os.listdir(path_to_mat))
subjects = defaultdict(list)
subject_fs = []
pattern = re.compile(r'^(SC4\d+_\d|S\d+_\d)')

for file in files:
    match = pattern.match(file)
    if match:
        subject_name = match.group(1)
        if subject_name not in subjects:
            if 'SC' in subject_name:
                fs = 100
            else:
                fs = 250
            subject_fs.append(fs)
        subjects[subject_name].append(file)

num_processes = mp.cpu_count()
print('Number of processes :', num_processes)

print(len(subjects.keys()))
print(len(subject_fs))

for (key, recording), fs in zip(subjects.items(), subject_fs):
    with h5py.File(hdf5_path, 'a')  as database:
        if key in list(database.keys()):
            print(f'{key} already exists, skipping...')
            continue
        else:
            print(f'Starting on subject: {key}')
            prepare_for_hdf5(key, recording, fs, path_to_mat, epoch_length, hdf5_path)

#Parallel(n_jobs=2, verbose=10)(delayed(prepare_for_hdf5)(key, recording, fs, path_to_mat, epoch_length, hdf5_path) for (key, recording), fs in zip(subjects.items(), subject_fs))
#results = Parallel(n_jobs=1, verbose = 10)(delayed(prepare_for_hdf5)(recording, fs, path_to_pt5, epoch_length) for recording in files)

# for result in results:
#     print(result)
#     update_hdf5(result, hdf5_path)

Number of processes : 12
251
251
Starting on subject: SC400_1
Subject name: SC400_1
